# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

This link doesn't work

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

In [ ]:
# If needed in a fresh environment, uncomment:
# %pip install pandas numpy scikit-learn matplotlib seaborn shap joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
DATA_PATH = "forestfires.csv"  # Keep the CSV beside this notebook.
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
display(df.describe(include="all").T)


In [ ]:
# Target distribution: zeros and skewness matter for interpreting this task.
print("Zero-area observations:", int((df["area"] == 0).sum()), "/", len(df))
print("Area skewness:", df["area"].skew())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["area"], bins=40, ax=axes[0])
axes[0].set(title="Burned area (original scale)", xlabel="Area (ha)")
sns.histplot(np.log1p(df["area"]), bins=40, ax=axes[1])
axes[1].set(title="Burned area after log1p", xlabel="log1p(area)")
plt.tight_layout()
plt.show()


In [ ]:
TARGET = "area"
X = df.drop(columns=TARGET)
y = df[TARGET].astype(float)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

def make_preprocessor(nonlinear=False):
    # Median imputation is included defensively; the supplied .names file says no
    # missing values are present. Categorical variables are one-hot encoded.
    if nonlinear:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("power", PowerTransformer(method="yeo-johnson", standardize=False)),
            ("scale", StandardScaler())
        ])
    else:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ], remainder="drop")

def make_estimator(preprocessor, regressor):
    # TTR ensures CV scoring and final predictions are evaluated in hectares.
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", regressor)
    ])
    return TransformedTargetRegressor(
        regressor=pipe, func=np.log1p, inverse_func=np.expm1,
        check_inverse=False
    )


In [ ]:
# Four combinations: simple/nonlinear preprocessing x Ridge/Extra Trees.
models = {
    "Simple + Ridge": (
        make_estimator(make_preprocessor(False), Ridge()),
        {"regressor__model__alpha": np.logspace(-3, 3, 30)}
    ),
    "Nonlinear + Ridge": (
        make_estimator(make_preprocessor(True), Ridge()),
        {"regressor__model__alpha": np.logspace(-3, 3, 30)}
    ),
    "Simple + ExtraTrees": (
        make_estimator(make_preprocessor(False), ExtraTreesRegressor(
            random_state=RANDOM_STATE, n_jobs=-1
        )),
        {
            "regressor__model__n_estimators": [200, 400, 600],
            "regressor__model__max_features": [0.6, 0.8, 1.0],
            "regressor__model__min_samples_leaf": [1, 2, 4, 8],
            "regressor__model__max_depth": [None, 5, 10, 20]
        }
    ),
    "Nonlinear + ExtraTrees": (
        make_estimator(make_preprocessor(True), ExtraTreesRegressor(
            random_state=RANDOM_STATE, n_jobs=-1
        )),
        {
            "regressor__model__n_estimators": [200, 400, 600],
            "regressor__model__max_features": [0.6, 0.8, 1.0],
            "regressor__model__min_samples_leaf": [1, 2, 4, 8],
            "regressor__model__max_depth": [None, 5, 10, 20]
        }
    )
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
searches = {}
rows = []
for name, (estimator, param_distributions) in models.items():
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_distributions,
        n_iter=12 if "ExtraTrees" in name else 20,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True,
        verbose=0
    )
    search.fit(X_train, y_train)
    searches[name] = search
    pred = np.clip(search.predict(X_test), 0, None)
    rows.append({
        "model": name,
        "best_cv_RMSE_ha": -search.best_score_,
        "test_RMSE_ha": mean_squared_error(y_test, pred) ** 0.5,
        "test_MAE_ha": mean_absolute_error(y_test, pred),
        "test_R2": r2_score(y_test, pred),
        "best_params": search.best_params_
    })
    print(name, "finished")
results = pd.DataFrame(rows).sort_values("test_RMSE_ha")
display(results.drop(columns="best_params"))
display(results[["model", "best_params"]])


In [ ]:
# Select using training-only cross-validation RMSE. The test set is used only once for final evaluation.
results = results.sort_values("best_cv_RMSE_ha").reset_index(drop=True)
best_name = results.iloc[0]["model"]
best_search = searches[best_name]
best_model = best_search.best_estimator_
best_pred = np.clip(best_model.predict(X_test), 0, None)

print("Selected by lowest training-CV RMSE:", best_name)
print("Test RMSE (ha):", mean_squared_error(y_test, best_pred) ** 0.5)
print("Test MAE (ha):", mean_absolute_error(y_test, best_pred))
print("Test R2 (context only; not the selection metric):", r2_score(y_test, best_pred))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, best_pred, alpha=0.7)
lim = max(float(y_test.max()), float(best_pred.max()))
ax.plot([0, lim], [0, lim], "--", color="black")
ax.set(xlabel="Observed area (ha)", ylabel="Predicted area (ha)",
       title=f"Test predictions: {best_name}")
plt.tight_layout()
plt.show()

residuals = y_test.to_numpy() - best_pred
sns.scatterplot(x=best_pred, y=residuals)
plt.axhline(0, linestyle="--", color="black")
plt.xlabel("Predicted area (ha)")
plt.ylabel("Residual (observed - predicted, ha)")
plt.title("Test residuals")
plt.show()


In [ ]:
# Model-agnostic SHAP on the original input columns, so this works for either Ridge or Extra Trees.
# The fitted predictor returns hectares; SHAP therefore explains predictions in hectares.
X_background = X_train.sample(min(80, len(X_train)), random_state=RANDOM_STATE)
X_explain = X_test.iloc[:min(40, len(X_test))]

def shap_predict(data):
    # Keep the original column names and categorical values for the sklearn pipeline.
    if isinstance(data, pd.DataFrame):
        frame = data
    else:
        frame = pd.DataFrame(data, columns=X.columns)
    return best_model.predict(frame)

explainer = shap.Explainer(shap_predict, X_background, algorithm="permutation")
shap_values = explainer(X_explain, max_evals=2 * X.shape[1] + 1)

# Global bar summary is robust to categorical input columns.
shap.plots.bar(shap_values, max_display=13)


In [ ]:
# Local waterfall explanation for one held-out test observation.
local_i = 0
shap.plots.waterfall(shap_values[local_i], max_display=15)
print("Observed area (ha):", float(y_test.iloc[local_i]))
print("Predicted area (ha):", float(best_pred[local_i]))


In [ ]:
# Global importance, already expressed at the original-variable level.
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_importance = pd.DataFrame({
    "variable": X.columns,
    "mean_abs_SHAP_ha": mean_abs_shap
}).sort_values("mean_abs_SHAP_ha", ascending=False)
display(shap_importance)

sns.barplot(data=shap_importance, x="mean_abs_SHAP_ha", y="variable")
plt.title("Global SHAP importance")
plt.xlabel("Mean absolute SHAP value (hectares)")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()


# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [ ]:
# Load the libraries as required.

from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import pandas as pd

# Create the data directory relative to the 05_src directory
data_dir = Path("data") / "fires"
data_dir.mkdir(parents=True, exist_ok=True)

# Download the UCI Forest Fires dataset
url = "https://archive.ics.uci.edu/static/public/162/forest+fires.zip"
zip_path = data_dir / "forestfires.zip"

if not (data_dir / "forestfires.csv").exists():
    urlretrieve(url, zip_path)

    # Extract the archive
    with ZipFile(zip_path, "r") as archive:
        archive.extractall(data_dir)

# Load the dataset
df = pd.read_csv(data_dir / "forestfires.csv")

# Inspect the data
print(f"Dataset shape: {df.shape}")
display(df.head())

# Column names and data types
display(df.info())

# Summary statistics
display(df.describe())

# Check for missing values
display(df.isnull().sum())

# Check the target variable
display(df["area"].describe())




X = df.drop(columns=["area"])
y = df["area"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Identify categorical and numerical columns
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", categorical_features)
print("Numerical features:", numeric_features)

In [ ]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


# Get X and Y

Create the features data frame and target data.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("forestfires.csv")

# Features: all columns except area
X = df.drop(columns=["area"])

# Target: burned forest area
y = df["area"]

# Display results
print("Features shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [ ]:
# Get X and y
X = fires_dt.drop(columns=['area'])
y = fires_dt['area']

# Check dimensions
print("Features shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer

# Identify column types
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns

print("Numerical variables:", list(numeric_features))
print("Categorical variables:", list(categorical_features))

In [ ]:
preproc1 = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder

# Identify numeric and categorical features
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Numeric preprocessing: scale and apply a non-linear transformation
numeric_transformer = Pipeline(steps=[
    ('power', PowerTransformer(method='yeo-johnson')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing: one-hot encoding
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Combine preprocessing steps
preproc1 = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline

preproc2 = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("power", PowerTransformer(method="yeo-johnson")),
            ("scaler", StandardScaler())
        ]), numeric_features),

        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Apply transformations
X_num_original = X[numeric_features]

X_num_transformed = preproc2.named_transformers_["num"].fit_transform(
    X_num_original
)

# Compare skewness
comparison = pd.DataFrame({
    "Original skewness": X_num_original.skew(),
    "Transformed skewness": pd.DataFrame(
        X_num_transformed,
        columns=numeric_features
    ).skew()
})

display(comparison)

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [ ]:
# Pipeline A = preproc1 + baseline
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Baseline model: Ridge Regression
ridge_pipeline = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('regressor', Ridge())
])

# Advanced model: Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
])

In [ ]:
# Pipeline B = preproc2 + baseline
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import root_mean_squared_error
import numpy as np

cv = KFold(n_splits=5, shuffle=True, random_state=42)

ridge_param_grid = {
    'regressor__alpha': [0.01, 0.1, 1, 10, 100]
}

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1
)

ridge_search.fit(X, y)

print("Best Ridge parameters:", ridge_search.best_params_)
print("Best Ridge CV RMSE:", -ridge_search.best_score_)

In [ ]:
# Pipeline C = preproc1 + advanced model
rf_param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [None, 5, 10],
    'regressor__min_samples_leaf': [1, 3, 5],
    'regressor__max_features': [0.7, 1.0]
}

rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1
)

rf_search.fit(X, y)

print("Best Random Forest parameters:", rf_search.best_params_)
print("Best Random Forest CV RMSE:", -rf_search.best_score_)


In [ ]:
# Pipeline D = preproc2 + advanced model

    results = pd.DataFrame({
    'Model': ['Ridge Regression', 'Random Forest'],
    'Best CV RMSE': [
        -ridge_search.best_score_,
        -rf_search.best_score_
    ]
}).sort_values('Best CV RMSE')

display(results)


best_search = min(
    [ridge_search, rf_search],
    key=lambda search: -search.best_score_
)

best_model = best_search.best_estimator_

print("Best model:", best_model.named_steps['regressor'])
print("Best CV RMSE:", -best_search.best_score_)


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

pipelines = {
    "ridge_preproc1": Pipeline([
        ("preprocessing", preproc1),
        ("regressor", Ridge())
    ]),
    "ridge_preproc2": Pipeline([
        ("preprocessing", preproc2),
        ("regressor", Ridge())
    ]),
    "rf_preproc1": Pipeline([
        ("preprocessing", preproc1),
        ("regressor", RandomForestRegressor(random_state=42, n_jobs=1))
    ]),
    "rf_preproc2": Pipeline([
        ("preprocessing", preproc2),
        ("regressor", RandomForestRegressor(random_state=42, n_jobs=1))
    ])
}

In [ ]:
param_grids = {
    "ridge_preproc1": {
        "regressor__alpha": [0.01, 0.1, 1, 10]
    },
    "ridge_preproc2": {
        "regressor__alpha": [0.01, 0.1, 1, 10]
    },
    "rf_preproc1": {
        "regressor__n_estimators": [100, 200],
        "regressor__max_depth": [None, 10]
    },
    "rf_preproc2": {
        "regressor__n_estimators": [100, 200],
        "regressor__max_depth": [None, 10]
    }
}

In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

searches = {}
results = []

for name, pipeline in pipelines.items():
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        refit=True
    )

    search.fit(X, y)
    searches[name] = search

    results.append({
        "Pipeline": name,
        "Best Parameters": search.best_params_,
        "Best CV RMSE": -search.best_score_
    })

results_df = pd.DataFrame(results).sort_values("Best CV RMSE")
display(results_df)

In [ ]:
best_name = results_df.iloc[0]["Pipeline"]
best_model = searches[best_name].best_estimator_

print("Best pipeline:", best_name)
print("Best parameters:", searches[best_name].best_params_)
print("Best CV RMSE:", -searches[best_name].best_score_)

# Evaluate

+ Which model has the best performance?  **The Nonlinear Preprocessing + Ridge Regression model has the best overall performance**


Model performance comparison
Model pipeline	CV RMSE (ha)	Test RMSE (ha)	Test MAE (ha)	Test R²
Nonlinear + Ridge	40.33	109.82	20.19	-0.0232
Simple + Ridge	40.36	109.80	20.11	-0.0227
Nonlinear + Extra Trees	40.53	109.87	19.80	-0.0241
Simple + Extra Trees	40.53	109.86	19.77	-0.0240





The Nonlinear Preprocessing + Ridge Regression pipeline is selected because it achieved the lowest cross-validation RMSE of approximately 40.33 hectares.

However, the test results show that all four models perform similarly and have negative R² values. This indicates that they do not predict the test data substantially better than simply predicting the average fire area.

The large difference between cross-validation RMSE and test RMSE also suggests that the models may struggle with the highly skewed distribution of burned areas, particularly very large fires.

Final answer: Choose Nonlinear Preprocessing + Ridge Regression based on cross-validation RMSE, but acknowledge that its predictive performance on unseen test data is weak.

# Export

+ Save the best performing model to a pickle file.

In [ ]:
import pickle
from pathlib import Path

# Define the output path
output_path = Path("best_forest_fire_model.pkl")

# Export the best model
with open(output_path, "wb") as file:
    pickle.dump(best_model, file)

print(f"Model saved to: {output_path.resolve()}")

In [ ]:
with open("best_forest_fire_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

# Generate predictions on new data
predictions = loaded_model.predict(X)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import root_mean_squared_error

# Create a held-out test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Re-tune the winning pipeline using training data only
final_search = GridSearchCV(
    estimator=pipelines[best_name],
    param_grid=param_grids[best_name],
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    refit=True
)

final_search.fit(X_train, y_train)
best_model = final_search.best_estimator_

# Evaluate on the untouched test set
y_pred = best_model.predict(X_test)

print("Selected pipeline:", best_name)
print("Test RMSE:", root_mean_squared_error(y_test, y_pred))

In [ ]:
# Separate the fitted preprocessing and regression steps
preprocessor = best_model.named_steps["preprocessing"]
regressor = best_model.named_steps["regressor"]

# Transform train and test data using the fitted preprocessor
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Get feature names after scaling, transformation and encoding
feature_names = preprocessor.get_feature_names_out()

# Convert transformed arrays to DataFrames for easier inspection
X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=X_train.index
)

X_test_transformed = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=X_test.index
)

# Use an appropriate SHAP explainer
if hasattr(regressor, "estimators_"):
    explainer = shap.TreeExplainer(regressor)
    shap_values = explainer(X_test_transformed)
else:
    explainer = shap.Explainer(
        regressor, X_train_transformed
    )
    shap_values = explainer(X_test_transformed)

print("SHAP values shape:", shap_values.values.shape)

*(Answer here.)*

The selected model is Nonlinear Preprocessing + Ridge Regression. I used this model to calculate SHAP values, with the explanations expressed in the original target unit: hectares burned. I selected the first observation in the test set, which is row 304 in the original dataset.

The SHAP explanation starts from a baseline prediction of approximately 2.74 hectares. The following features had the largest effects on this particular prediction:
Feature	SHAP contribution	Effect
Month: May	+45.66 ha	Increased prediction
DC	+15.73 ha	Increased prediction
DMC	-12.89 ha	Decreased prediction
Relative humidity	+8.17 ha	Increased prediction
Temperature	-4.97 ha	Decreased prediction
Month: September	-3.96 ha	Decreased prediction
Month: August	+3.87 ha	Increased prediction
Month: December	-3.21 ha	Decreased prediction
ISI	+3.06 ha	Increased prediction

Positive SHAP values push the prediction higher, while negative values push it lower. The contributions add to approximately 59.05 hectares.

Interpretation: The month of May was the largest contributor to the model's prediction, followed by DC and DMC. Although DMC and temperature reduced the prediction, the positive contributions—particularly May and DC—were larger overall.

An important limitation is that the model predicted approximately 59 hectares when the actual burned area was zero. This illustrates that the model can make substantial errors for individual observations.
--> Global explanation: feature importance across the training set
I calculated the mean absolute SHAP value for each original feature across all 413 training observations. For categorical variables, I combined the SHAP contributions of their one-hot encoded categories.

Which features would I remove?
Based on the global SHAP results, I would consider removing the following features first:
Feature	Reason
Y	Extremely low SHAP importance
Rain	Low SHAP importance
ISI	Low SHAP importance
Wind	Low SHAP importance
FFMC	Relatively low SHAP importance

I would retain Month, DC, DMC, Temperature, Day, RH and X initially because they have higher SHAP importance.

However, I would not remove these features solely because their SHAP values are low. A feature with low individual importance may still provide useful information when combined with other features.
to test whether removing features improves performance?
I would use a feature ablation experiment:
1. Establish the original model's performance using all 12 features.
2. Create a reduced dataset without the five proposed features.
3. Rebuild the preprocessing and regression pipeline using only the remaining features.
4. Tune the reduced model using the same hyperparameter search and cross-validation procedure.
5. Compare the cross-validation RMSE of the original and reduced models.
6. Evaluate the selected reduced model on the same untouched test set.
The primary metric would be RMSE in hectares. Lower RMSE indicates better predictive performance.

For a more reliable comparison, I would use repeated cross-validation with the same folds for both models.
Result	Interpretation
Reduced model has lower RMSE	Removing features may improve performance
Reduced model has similar RMSE	Features may be unnecessary
Reduced model has higher RMSE	Removed features were contributing useful information

I would only keep the reduced model if it performs as well as or better than the original model consistently across validation folds.
Overall conclusion
The SHAP analysis suggests that Month, DC and DMC are the most influential features in the selected Ridge regression model. Y, Rain, ISI, Wind and FFMC are potential candidates for removal because they have relatively low global importance.

I would test feature removal using cross-validation RMSE and a final held-out test set rather than relying on SHAP importance alone. This would help determine whether removing less important features actually improves the model's predictive performance.

In [ ]:
For this section, you’ll use SHAP (SHapley Additive exPlanations) to explain the best-performing model selected by your grid search. You need both a local explanation (one test observation) and a global explanation (feature importance across the training data).

Because your best model is a pipeline, SHAP should explain the features after preprocessing while still displaying meaningful original feature names wherever possible. For a tree-based Random Forest, shap.TreeExplainer is a good choice.






## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.